In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load ranked baseline (EGFR)
ranked_baseline = pd.read_parquet("../../processed/ENSG00000146648_composite_baseline.parquet")

# Load the cleaned contextual layers
signatures   = pd.read_parquet("../../processed/signatures_clean.parquet")
metabolomics = pd.read_parquet("../../processed/metabolomics_clean.parquet")
mirna        = pd.read_parquet("../../processed/mirna_clean.parquet")

print("Ranked Baseline:", ranked_baseline.shape)
print("Signatures:", signatures.shape)
print("Metabolomics:", metabolomics.shape)
print("miRNA:", mirna.shape)

Ranked Baseline: (1581, 30)
Signatures: (1955, 6)
Metabolomics: (928, 227)
miRNA: (734, 952)


In [2]:
# check how cell line IDs are stored before writing the exclusion logic
print("--- Metabolomics ---")
print("Index name:", metabolomics.index.name)
print("First 5 columns:", metabolomics.columns.tolist()[:5])

print("\n--- miRNA ---")
print("Index name:", mirna.index.name)
print("First 5 columns:", mirna.columns.tolist()[:5])

--- Metabolomics ---
Index name: None
First 5 columns: ['CCLE_ID', 'DepMap_ID', '2-aminoadipate', '3-phosphoglycerate', 'alpha-glycerophosphate']

--- miRNA ---
Index name: miRNA
First 5 columns: ['ACH-000698', 'ACH-000489', 'ACH-000431', 'ACH-000707', 'ACH-000509']


In [3]:
fact_expr_dm = pd.read_csv("../../data/AZ_harmonized_data/fact_expression_depmap.csv")
fact_mut     = pd.read_csv("../../data/AZ_harmonized_data/fact_mutations.csv")
fact_fus     = pd.read_csv("../../data/AZ_harmonized_data/fact_fusions.csv")

In [53]:
def apply_full_exclusion(ranked_df, tables, criteria=None):
    """
    Filters cell lines against user-supplied exclusion criteria.

    tables   : dict of the source tables, e.g. {"signatures": ..., "metabolomics": ..., "mirna": ...}
    criteria : dict of exclusion rules. Supported keys:
                 "msi_max"     : float, exclude MSIScore above this
                 "cin_max"     : float, exclude CIN above this
                 "metabolite"  : (name, threshold) tuple
                 "mirna"       : (name, threshold) tuple
                 "gene_expressed"    : (ensembl_id, percentile) tuple
                 "damaging_mutation" : ensembl_id
                 "gene_in_fusion"    : ensembl_id
                 
    Signature thresholds use established biological cutoffs. Metabolite and miRNA
    thresholds have no default: there is no biologically defensible universal
    cutoff for "too high" across different metabolites or miRNAs, so the caller
    must supply an absolute value.
    """
    criteria = criteria or {}
    sig_df = tables["signatures"]

    merged = pd.merge(ranked_df, sig_df, left_on="ACH_ID", right_index=True, how="left")
    exclude_cond = pd.Series(False, index=merged.index)

    # genomic instability, established cutoffs
    if "msi_max" in criteria:
        exclude_cond |= (merged["MSIScore"] > criteria["msi_max"]).fillna(False).astype(bool)
    if "cin_max" in criteria:
        exclude_cond |= (merged["CIN"] > criteria["cin_max"]).fillna(False).astype(bool)

    # metabolite, threshold required
    if "metabolite" in criteria:
        name, threshold = criteria["metabolite"]
        metab_df = tables.get("metabolomics")
        if metab_df is None or name not in metab_df.columns:
            raise ValueError(f"Metabolite '{name}' not found in metabolomics data.")
        if threshold is None:
            raise ValueError(f"No threshold supplied for metabolite '{name}'.")
        high = metab_df.set_index("DepMap_ID")[name] > threshold
        exclude_cond |= merged["ACH_ID"].map(high).eq(True)

    # miRNA, threshold required
    if "mirna" in criteria:
        name, threshold = criteria["mirna"]
        mirna_df = tables.get("mirna")
        if mirna_df is None or name not in mirna_df.index:
            raise ValueError(f"miRNA '{name}' not found in miRNA data.")
        if threshold is None:
            raise ValueError(f"No threshold supplied for miRNA '{name}'.")
        high = mirna_df.loc[name] > threshold
        exclude_cond |= merged["ACH_ID"].map(high).eq(True)

    # exclude cell lines expressing an unwanted gene above a threshold percentile
    if "gene_expressed" in criteria:
        gene, pct_threshold = criteria["gene_expressed"]
        expr_df = tables.get("expression")
        if expr_df is None:
            raise ValueError("Expression table not supplied.")
        sub = expr_df[expr_df["ensembl_id"] == gene][["ACH_ID", "log2_tpm_plus1"]]
        sub = sub.groupby("ACH_ID")["log2_tpm_plus1"].median()
        high = sub.rank(pct=True) * 100 > pct_threshold
        exclude_cond |= merged["ACH_ID"].map(high).eq(True)

    # exclude cell lines with a damaging mutation in a given gene
    if "damaging_mutation" in criteria:
        gene = criteria["damaging_mutation"]
        mut_df = tables.get("mutations")
        if mut_df is None:
            raise ValueError("Mutations table not supplied.")
        hit = mut_df[(mut_df["ensembl_id"] == gene) &
                     (mut_df["is_damaging"] | mut_df["is_lof"] | mut_df["is_hotspot"])]["ach_id"].unique()
        exclude_cond |= merged["ACH_ID"].isin(hit)

    # exclude cell lines where a given gene participates in any fusion
    if "gene_in_fusion" in criteria:
        gene = criteria["gene_in_fusion"]
        fus_df = tables.get("fusions")
        if fus_df is None:
            raise ValueError("Fusions table not supplied.")
        g1 = fus_df["gene1_ensg"].str.extract(r"(ENSG\d+)")[0]
        g2 = fus_df["gene2_ensg"].str.extract(r"(ENSG\d+)")[0]
        hit = fus_df.loc[(g1 == gene) | (g2 == gene), "ach_id"].unique()
        exclude_cond |= merged["ACH_ID"].isin(hit)

    # coverage flags: distinguish "checked and clean" from "not checked"
    if "mutations" in tables:
        merged["has_mutation_data"] = merged["ACH_ID"].isin(tables["mutations"]["ach_id"])
    if "fusions" in tables:
        merged["has_fusion_data"] = merged["ACH_ID"].isin(tables["fusions"]["ach_id"])
        
    viable = merged[~exclude_cond].copy()
    print(f"Original: {len(merged)} | Excluded: {exclude_cond.sum()} | Remaining: {len(viable)}")
    return viable
    
tables = {"signatures": signatures, "metabolomics": metabolomics, "mirna": mirna,
          "expression": fact_expr_dm, "mutations": fact_mut, "fusions": fact_fus}

viable_candidates = apply_full_exclusion(ranked_baseline, tables,
                                          criteria={"msi_max": 3.0, "cin_max": 0.5})

cols_to_show = ["ACH_ID", "cell_line_name", "evidence_score", "MSIScore", "CIN", "confidence_score"]
display(viable_candidates[cols_to_show].head(10))

Original: 1581 | Excluded: 978 | Remaining: 603


,ACH_ID,cell_line_name,evidence_score,MSIScore,CIN,confidence_score
8,ACH-000528,ABC1,92.656002,1.84,NaN,0.938606
11,ACH-000606,PECAPJ34CLONEC12,94.313810,2.97,0.494775,0.895543
14,ACH-000222,ASPC1,91.668152,2.61,NaN,0.916126
16,ACH-000832,CAL27,94.043726,1.28,NaN,0.888914
17,ACH-000247,OCUM1,85.851502,1.91,0.408069,0.926013
21,ACH-000511,CALU1,86.978347,0.83,0.435122,0.898058
24,ACH-000768,MDAMB231,88.080164,2.14,0.447449,0.865674
27,ACH-000040,U118MG,82.243814,2.56,NaN,0.890342
29,ACH-000367,NCIH226,84.949704,1.20,NaN,0.853426
46,ACH-000260,SKNAS,75.181627,1.73,NaN,0.914666


In [54]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


def build_feature_matrix(sig_df, metab_df, mirna_df):
    """
    Combine the three contextual layers into one scaled per-cell-line matrix.
    Standardised so the miRNA block (values up to ~188,000) does not dominate
    signatures (~0-3) and metabolites (~4-7) in the distance calculation.
    """
    sig_c   = sig_df.reset_index().rename(columns={"ModelID": "ACH_ID"})
    metab_c = metab_df.rename(columns={"DepMap_ID": "ACH_ID"}).drop(columns=["CCLE_ID"])
    mirna_c = mirna_df.T
    mirna_c.index.name = "ACH_ID"
    mirna_c = mirna_c.reset_index()

    master = (sig_c.merge(metab_c, on="ACH_ID", how="inner")
                   .merge(mirna_c, on="ACH_ID", how="inner"))

    feat_cols = [c for c in master.columns if c != "ACH_ID"]

    X = master[feat_cols].astype(float)
    X = X.fillna(X.median())                      # median, not 0: keeps missing near-neutral
    master[feat_cols] = StandardScaler().fit_transform(X)

    return master, feat_cols


def recommend_similar(target_ach, viable_df, master, feat_cols, top_n=10):
    """
    Find the most similar viable cell lines to a target, across the scaled
    contextual feature space.
    """
    if target_ach not in master["ACH_ID"].values:
        return f"Cannot compute: {target_ach} missing from one or more omics layers."

    target_vec = master.loc[master["ACH_ID"] == target_ach, feat_cols]

    pool = viable_df[["ACH_ID", "cell_line_name", "evidence_score"]].merge(
        master, on="ACH_ID", how="inner")
    pool = pool[pool["ACH_ID"] != target_ach].copy()

    if pool.empty:
        return "No viable cell lines remaining with matching omics data."

    pool["similarity_score"] = cosine_similarity(target_vec, pool[feat_cols])[0]

    return (pool.sort_values("similarity_score", ascending=False)
                .head(top_n)[["ACH_ID", "cell_line_name", "evidence_score", "similarity_score"]])


master, feat_cols = build_feature_matrix(signatures, metabolomics, mirna)
print(f"{len(master)} cell lines with all three layers | {len(feat_cols)} features")

target_cell_line = "ACH-000431"
twins = recommend_similar(target_cell_line, viable_candidates, master, feat_cols)
print(f"\nTop alternatives for {target_cell_line}:")
display(twins)

895 cell lines with all three layers | 965 features

Top alternatives for ACH-000431:


,ACH_ID,cell_line_name,evidence_score,similarity_score
226,ACH-000816,NCIH524,23.473501,0.339610
262,ACH-000382,CORL24,16.166329,0.313536
175,ACH-000366,SKNDZ,38.713788,0.279257
184,ACH-000259,KELLY,36.470068,0.227479
157,ACH-000399,NCIH2196,47.248141,0.201650
224,ACH-000052,A673,19.271909,0.196167
127,ACH-000341,SKNFI,46.312940,0.179256
131,ACH-000293,KLE,42.952485,0.176289
239,ACH-000151,JM1,15.602299,0.169513
106,ACH-000790,SHP77,39.922587,0.169041


In [55]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def lineage_test(matrix, feat_cols, label, n_pairs=2000, seed=0):
    rng = np.random.default_rng(seed)
    m = matrix.dropna(subset=["lineage"]).reset_index(drop=True)
    X = m[feat_cols].fillna(0).values
    sims = cosine_similarity(X)
    same, diff = [], []
    for _ in range(n_pairs):
        i, j = rng.integers(0, len(m), 2)
        if i == j:
            continue
        (same if m.loc[i, "lineage"] == m.loc[j, "lineage"] else diff).append(sims[i, j])
    print(f"{label}")
    print(f"  same lineage: mean {np.mean(same):.4f}  (n={len(same)})")
    print(f"  diff lineage: mean {np.mean(diff):.4f}  (n={len(diff)})")
    print(f"  gap: {np.mean(same) - np.mean(diff):+.4f}")

# unscaled matrix, built explicitly so the comparison doesn't depend on execution order
sig_c   = signatures.reset_index().rename(columns={"ModelID": "ACH_ID"})
metab_c = metabolomics.rename(columns={"DepMap_ID": "ACH_ID"}).drop(columns=["CCLE_ID"])
mirna_c = mirna.T; mirna_c.index.name = "ACH_ID"; mirna_c = mirna_c.reset_index()
raw_master = (sig_c.merge(metab_c, on="ACH_ID", how="inner")
                   .merge(mirna_c, on="ACH_ID", how="inner")
                   .merge(ranked_baseline[["ACH_ID", "lineage"]], on="ACH_ID", how="left"))

scaled_master = master.merge(ranked_baseline[["ACH_ID", "lineage"]], on="ACH_ID", how="left")

lineage_test(raw_master, feat_cols, "raw features")
lineage_test(scaled_master, feat_cols, "z-scored features")

raw features
  same lineage: mean 0.6331  (n=154)
  diff lineage: mean 0.5716  (n=1846)
  gap: +0.0615
z-scored features
  same lineage: mean 0.0798  (n=154)
  diff lineage: mean -0.0005  (n=1846)
  gap: +0.0803


In [56]:
# check metabolite range before choosing a threshold
metab_col = "lactate" if "lactate" in metabolomics.columns else metabolomics.columns[2]
print(metab_col)
print(metabolomics[metab_col].describe())

# same for a miRNA
mirna_row = "hsa-miR-21" if "hsa-miR-21" in mirna.index else mirna.index[0]
print("\n", mirna_row)
print(mirna.loc[mirna_row].describe())

lactate
count    928.000000
mean       5.820187
std        0.269868
min        4.110226
25%        5.694144
50%        5.859198
75%        5.996382
max        6.631586
Name: lactate, dtype: float64

 hsa-miR-21
count       952.000000
mean       6172.787598
std        9810.112305
min          14.450000
25%        1337.745056
50%        3540.315063
75%        7368.340210
max      188089.265625
Name: hsa-miR-21, dtype: float64


In [57]:
# demonstration thresholds only - chosen from the observed distributions to show
# the branches execute, not because 6.0 lactate or 7400 miR-21 is biologically meaningful

viable_metab = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5, "metabolite": ("lactate", 6.0)})

viable_mirna = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5, "mirna": ("hsa-miR-21", 7400)})

viable_both = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5,
              "metabolite": ("lactate", 6.0), "mirna": ("hsa-miR-21", 7400)})

# confirm the guard rail fires when a threshold is missing
try:
    apply_full_exclusion(ranked_baseline, tables,
        criteria={"metabolite": ("lactate", None)})
except ValueError as e:
    print("\nValueError raised as expected:", e)

Original: 1581 | Excluded: 1040 | Remaining: 541
Original: 1581 | Excluded: 1055 | Remaining: 526
Original: 1581 | Excluded: 1103 | Remaining: 478

ValueError raised as expected: No threshold supplied for metabolite 'lactate'.


In [58]:
print("mutations:", fact_mut.shape, fact_mut.columns.tolist())
print("fusions:",   fact_fus.shape, fact_fus.columns.tolist())

mutations: (193318, 15) ['ach_id', 'ensembl_id', 'hugo_symbol', 'chrom', 'pos', 'ref', 'alt', 'variant_type', 'variant_info', 'protein_change', 'is_driver', 'allele_freq', 'is_hotspot', 'is_damaging', 'is_lof']
fusions: (43095, 13) ['ach_id', 'gene1_ensg', 'gene2_ensg', 'gene1_hugo', 'gene2_hugo', 'fusion_name', 'ffpm', 'confidence', 'supporting_reads', 'split_reads1', 'split_reads2', 'discordant_mates', 'reading_frame']


In [59]:
apply_full_exclusion(ranked_baseline, tables, criteria={
    "damaging_mutation": "ENSG00000133703",   # KRAS
    "gene_in_fusion": "ENSG00000146648",      # EGFR, the target gene
})

Original: 1581 | Excluded: 80 | Remaining: 1501


,ACH_ID,depmap_value,depmap_spread,depmap_n_reps,depmap_percentile,hpa_value,hpa_spread,hpa_n_reps,hpa_percentile,geo_value,...,primary_disease,lineage,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy,has_mutation_data,has_fusion_data
0,ACH-000777,8.359530,0.0,1.0,99.256254,8.434211,0.0,1.0,99.457014,9.876311,...,Esophageal Cancer,esophagus,1.49,0.518947,1.0,0.733042,3.459052,18.0,True,True
2,ACH-000778,7.691464,0.0,1.0,98.174442,7.553053,0.0,1.0,97.647059,8.871437,...,Head and Neck Cancer,upper_aerodigestive,1.57,0.190249,1.0,0.573404,2.764623,20.0,False,False
3,ACH-000012,9.673645,0.0,1.0,99.864773,9.488443,0.0,1.0,99.728507,9.845259,...,Lung Cancer,lung,1.67,0.573864,1.0,0.642095,2.975856,20.0,True,True
4,ACH-000869,8.062046,0.0,1.0,98.985801,8.173427,0.0,1.0,99.366516,10.177364,...,Lung Cancer,lung,2.42,0.348272,1.0,0.715872,4.398623,36.0,True,True
5,ACH-000109,8.925347,0.0,1.0,99.526707,8.786923,0.0,1.0,99.547511,9.791943,...,Lung Cancer,lung,0.93,0.282786,1.0,0.755313,3.976113,29.0,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1576,ACH-002921,0.000000,0.0,1.0,1.791751,NaN,NaN,NaN,NaN,NaN,...,None,None,2.07,0.134004,0.0,0.190332,2.278344,8.0,False,False
1577,ACH-001573,0.000000,0.0,1.0,1.791751,NaN,NaN,NaN,NaN,NaN,...,Leukemia,blood,3.22,NaN,NaN,NaN,NaN,NaN,False,False
1578,ACH-001574,0.000000,0.0,1.0,1.791751,NaN,NaN,NaN,NaN,NaN,...,Leukemia,blood,2.66,0.015127,0.0,0.190192,2.158543,6.0,False,False
1579,ACH-001577,0.000000,0.0,1.0,1.791751,NaN,NaN,NaN,NaN,NaN,...,Leukemia,blood,1.83,0.116023,1.0,0.657283,4.945604,31.0,False,False


In [60]:
# mutation coverage: only a fraction of ranked cell lines have mutation data,
# so exclusion can only act on those - the rest are unmeasured, not confirmed clean
print("cell lines with mutation data:", fact_mut["ach_id"].nunique())
print("cell lines in ranking:", ranked_baseline["ACH_ID"].nunique())

cell lines with mutation data: 365
cell lines in ranking: 1581


In [61]:
# exclude cell lines expressing ERBB2 above the 90th percentile
result_expr = apply_full_exclusion(ranked_baseline, tables,
    criteria={"gene_expressed": ("ENSG00000141736", 90)})

# sanity check: a known HER2-high line should be gone
print("HCC1954 removed:", "ACH-000859" not in set(result_expr["ACH_ID"]))

Original: 1581 | Excluded: 148 | Remaining: 1433
HCC1954 removed: True


In [62]:
result = apply_full_exclusion(ranked_baseline, tables,
                              criteria={"damaging_mutation": "ENSG00000133703"})
print(result["has_mutation_data"].value_counts())

Original: 1581 | Excluded: 73 | Remaining: 1508
has_mutation_data
False    1216
True      292
Name: count, dtype: int64
